# Cohort Characteristics with OMOPy

This notebook demonstrates `omopy.characteristics` — summarising, tabling, and plotting cohort characteristics.

We cover:
- **Summarise** functions that return `SummarisedResult` objects
- **Table** functions that render GT or Polars tables
- **Plot** functions that produce figures
- Working with `SummarisedResult` (tidy, pivot, filter, suppress)
- Mock data for testing

## 1. Configuration

In [1]:
DB_PATH = "../data/synthea.duckdb"
CDM_SCHEMA = "base"
WRITE_SCHEMA = "main"

## 2. Setup — Connect and Generate Cohort

Before computing characteristics we need a CDM connection and a cohort.
We generate a concept-based cohort with two conditions from the Synthea database (27 persons).

In [2]:
from omopy.connector import cdm_from_con, generate_concept_cohort_set
from omopy.generics import Codelist

cdm = cdm_from_con(DB_PATH, cdm_schema=CDM_SCHEMA, write_schema=WRITE_SCHEMA)
print(cdm)

CdmReference(name='dbt-synthea', version=5.4, source=duckdb, tables=36)


In [3]:
concept_set = Codelist({"sinusitis": [40481087], "hypertension": [320128]})
cdm = generate_concept_cohort_set(cdm, concept_set, name="conditions")
cohort = cdm["conditions"]
print(cohort)

CohortTable('conditions', source='duckdb', cohorts=2)


## 3. Summarise Characteristics

`summarise_characteristics` computes demographics and clinical features for each cohort.
It takes a single `cohort` argument (a `CohortTable` which carries a reference to its CDM).
The `estimates` parameter is a dict mapping variable types to tuples of estimate names.

In [4]:
from omopy.characteristics import summarise_characteristics

char_result = summarise_characteristics(
    cohort,
    estimates={
        "numeric": ("median", "q25", "q75", "mean", "sd"),
        "categorical": ("count", "percentage"),
    },
)
print(type(char_result))
print(char_result)

<class 'omopy.generics.summarised_result.SummarisedResult'>
SummarisedResult(68 rows, 1 result_id(s))


In [5]:
# Inspect the underlying data
print(char_result.data)

shape: (68, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ result_id ┆ cdm_name  ┆ group_nam ┆ group_lev ┆ … ┆ variable_ ┆ estimate_ ┆ estimate_ ┆ estimate │
│ ---       ┆ ---       ┆ e         ┆ el        ┆   ┆ level     ┆ name      ┆ type      ┆ _value   │
│ i64       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1         ┆ dbt-synth ┆ cohort_na ┆ sinusitis ┆ … ┆           ┆ count     ┆ integer   ┆ 3        │
│           ┆ ea        ┆ me        ┆           ┆   ┆           ┆           ┆           ┆          │
│ 1         ┆ dbt-synth ┆ cohort_na ┆ sinusitis ┆ … ┆           ┆ count     ┆ integer   ┆ 3        │
│           ┆ ea        ┆ me        ┆           ┆   ┆           ┆          

## 4. Table & Plot Characteristics

Render the summarised result as a GT table, a Polars DataFrame, or a plot.

In [6]:
from omopy.characteristics import table_characteristics, plot_characteristics

# GT table (rich HTML rendering)
gt_table = table_characteristics(char_result, type="gt")
gt_table

GT(_tbl_data=         variable_name variable_level       estimate_name  \
0                  Sex           Male               N (%)   
1                  Sex         Female               N (%)   
2                  Sex         Female               N (%)   
3                  Sex           Male               N (%)   
4       Number records                                  N   
5      Number subjects                                  N   
6                  Sex           Male                   N   
7                  Sex         Female                   N   
8       Number records                                  N   
9      Number subjects                                  N   
10                 Sex         Female                   N   
11                 Sex           Male                   N   
12   Prior observation                 Median [Q25 - Q75]   
13  Future observation                 Median [Q25 - Q75]   
14                 Age                 Median [Q25 - Q75]   
15      Days in cohort                 Median [Q25 - Q75]   
16   Prior observation                 Median [Q25 - Q75]   
17  Future observation                 Median [Q25 - Q75]   
18                 Age                 Median [Q25 - Q75]   
19      Days in cohort                 Median [Q25 - Q75]   
20   Prior observation                          Mean (SD)   
21  Future observation                          Mean (SD)   
22                 Age                          Mean (SD)   
23      Days in cohort                          Mean (SD)   
24   Prior observation                          Mean (SD)   
25  Future observation                          Mean (SD)   
26                 Age                          Mean (SD)   
27      Days in cohort                          Mean (SD)   
28   Prior observation                              Range   
29  Future observation                              Range   
30                 Age                              Range   
31      Days in cohort                              Range   
32   Prior observation                              Range   
33  Future observation                              Range   
34                 Age                              Range   
35      Days in cohort                              Range   

   [header_name]cdm_name\n[header_level]dbt-synthea   cohort_name  
0                                         2 (66.7%)     sinusitis  
1                                         1 (33.3%)     sinusitis  
2                                         3 (50.0%)  hypertension  
3                                         3 (50.0%)  hypertension  
4                                                 3     sinusitis  
5                                                 3     sinusitis  
6                                                 2     sinusitis  
7                                                 1     sinusitis  
8                                                 6  hypertension  
9                                                 6  hypertension  
10                                                3  hypertension  
11                                                3  hypertension  
12                14,782.00 [14,782.00 - 14,943.00]     sinusitis  
13                         131.00 [131.00 - 268.00]     sinusitis  
14                            51.00 [51.00 - 54.00]     sinusitis  
15                         132.00 [132.00 - 269.00]     sinusitis  
16                       3,970.50 [0.00 - 6,166.00]  hypertension  
17                 10,084.00 [7,420.00 - 12,614.00]  hypertension  
18                            18.00 [18.00 - 18.00]  hypertension  
19                 10,085.00 [7,421.00 - 12,615.00]  hypertension  
20                               14,484.67 (659.36)     sinusitis  
21                                   160.67 (96.00)     sinusitis  
22                                     48.67 (6.81)     sinusitis  
23                                   161.67 (96.00)     sinusitis  
24                           

In [7]:
# Polars DataFrame table
pl_table = table_characteristics(char_result, type="polars")
print(pl_table)

shape: (36, 5)
┌────────────────────┬────────────────┬───────────────┬───────────────────────┬──────────────┐
│ variable_name      ┆ variable_level ┆ estimate_name ┆ [header_name]cdm_name ┆ cohort_name  │
│ ---                ┆ ---            ┆ ---           ┆ [header_…             ┆ ---          │
│ str                ┆ str            ┆ str           ┆ ---                   ┆ str          │
│                    ┆                ┆               ┆ str                   ┆              │
╞════════════════════╪════════════════╪═══════════════╪═══════════════════════╪══════════════╡
│ Sex                ┆ Male           ┆ N (%)         ┆ 2 (66.7%)             ┆ sinusitis    │
│ Sex                ┆ Female         ┆ N (%)         ┆ 1 (33.3%)             ┆ sinusitis    │
│ Sex                ┆ Female         ┆ N (%)         ┆ 3 (50.0%)             ┆ hypertension │
│ Sex                ┆ Male           ┆ N (%)         ┆ 3 (50.0%)             ┆ hypertension │
│ Number records     ┆             

In [8]:
# Plot characteristics
fig = plot_characteristics(char_result)
fig

## 5. Cohort Count

`summarise_cohort_count` returns person and record counts per cohort.

In [9]:
from omopy.characteristics import (
    summarise_cohort_count,
    table_cohort_count,
    plot_cohort_count,
)

count_result = summarise_cohort_count(cohort)
print(count_result)

SummarisedResult(4 rows, 1 result_id(s))


In [10]:
table_cohort_count(count_result, type="gt")

GT(_tbl_data=      CDM name    variable_name estimate_name  \
0  dbt-synthea   Number records             N   
1  dbt-synthea  Number subjects             N   

  [header_name]cohort_name\n[header_level]hypertension  \
0                                                  6     
1                                                  6     

  [header_name]cohort_name\n[header_level]sinusitis  
0                                                 3  
1                                                 3  , _body=<great_tables._gt_data.Body object at 0x72c90d781a70>, _boxhead=Boxhead([ColInfo(var='CDM name', type=<ColInfoTypeEnum.default: 1>, column_label='CDM name', column_align='left', column_width=None), ColInfo(var='variable_name', type=<ColInfoTypeEnum.default: 1>, column_label='variable_name', column_align='left', column_width=None), ColInfo(var='estimate_name', type=<ColInfoTypeEnum.default: 1>, column_label='estimate_name', column_align='left', column_width=None), ColInfo(var='[header_name]cohort_name\n[header_level]hypertension', type=<ColInfoTypeEnum.default: 1>, column_label='hypertension', column_align='right', column_width=None), ColInfo(var='[header_name]cohort_name\n[header_level]sinusitis', type=<ColInfoTypeEnum.default: 1>, column_label='sinusitis', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x72c90d614cd0>, _spanners=Spanners([SpannerInfo(spanner_id='cohort_name', spanner_level=0, spanner_label='cohort_name', spanner_units=None, spanner_pattern=None, vars=['[header_name]cohort_name\n[header_level]hypertension', '[header_name]cohort_name\n[header_level]sinusitis'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x72c90d615090>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x72c90d781cd0>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='CDM name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='estimate_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]cohort_name\n[header_level]hypertension', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]cohort_name\n[header_level]sinusitis', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='CDM name', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_name', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='estimate_name', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]cohort_name\n[header_level]hypertension', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(lo

In [11]:
fig = plot_cohort_count(count_result)
fig

## 6. Cohort Attrition

`summarise_cohort_attrition` shows how many subjects are lost at each inclusion step.

In [12]:
from omopy.characteristics import (
    summarise_cohort_attrition,
    table_cohort_attrition,
    plot_cohort_attrition,
)

attrition_result = summarise_cohort_attrition(cohort)
print(attrition_result)

SummarisedResult(8 rows, 1 result_id(s))


In [13]:
# Note: GT rendering for attrition tables has a known issue with cdm_name.
# We use try/except to gracefully fall back to the polars format.
try:
    display(table_cohort_attrition(attrition_result, type="gt"))
except (KeyError, Exception) as e:
    print(f"GT rendering unavailable ({e}); falling back to polars format:")
    print(table_cohort_attrition(attrition_result, type="polars"))

GT(_tbl_data=      CDM name [header_name]variable_name\n[header_level]excluded_records  \
0  dbt-synthea                                                  0           
1  dbt-synthea                                                  0           

    cohort_name                     reason  \
0     sinusitis  Initial qualifying events   
1  hypertension  Initial qualifying events   

  [header_name]variable_name\n[header_level]excluded_subjects  \
0                                                  0            
1                                                  0            

  [header_name]variable_name\n[header_level]number_records  \
0                                                  3         
1                                                  6         

  [header_name]variable_name\n[header_level]number_subjects  
0                                                  3         
1                                                  6         , _body=<great_tables._gt_data.Body object at 0x72c90d8cf9b0>, _boxhead=Boxhead([ColInfo(var='CDM name', type=<ColInfoTypeEnum.default: 1>, column_label='CDM name', column_align='left', column_width=None), ColInfo(var='[header_name]variable_name\n[header_level]excluded_records', type=<ColInfoTypeEnum.default: 1>, column_label='excluded_records', column_align='right', column_width=None), ColInfo(var='[header_name]variable_name\n[header_level]excluded_subjects', type=<ColInfoTypeEnum.default: 1>, column_label='excluded_subjects', column_align='right', column_width=None), ColInfo(var='[header_name]variable_name\n[header_level]number_records', type=<ColInfoTypeEnum.default: 1>, column_label='number_records', column_align='right', column_width=None), ColInfo(var='[header_name]variable_name\n[header_level]number_subjects', type=<ColInfoTypeEnum.default: 1>, column_label='number_subjects', column_align='right', column_width=None), ColInfo(var='cohort_name', type=<ColInfoTypeEnum.default: 1>, column_label='cohort_name', column_align='left', column_width=None), ColInfo(var='reason', type=<ColInfoTypeEnum.default: 1>, column_label='reason', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x72c90d614e10>, _spanners=Spanners([SpannerInfo(spanner_id='variable_name', spanner_level=0, spanner_label='variable_name', spanner_units=None, spanner_pattern=None, vars=['[header_name]variable_name\n[header_level]excluded_records', '[header_name]variable_name\n[header_level]excluded_subjects', '[header_name]variable_name\n[header_level]number_records', '[header_name]variable_name\n[header_level]number_subjects'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x72c90d7823f0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x72c90d7ae0f0>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='CDM name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]variable_name\n[header_level]excluded_records', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='cohort_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='reason', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]variable_name\n[header_level]excluded_subjects', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]variable_name\n[header_level]number_records', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=No

In [14]:
fig = plot_cohort_attrition(attrition_result)
fig

## 7. Cohort Timing

`summarise_cohort_timing` describes the time between cohort entry and exit dates.

In [15]:
from omopy.characteristics import (
    summarise_cohort_timing,
    table_cohort_timing,
    plot_cohort_timing,
)

timing_result = summarise_cohort_timing(cohort)
print(timing_result)

SummarisedResult(14 rows, 1 result_id(s))


In [16]:
table_cohort_timing(timing_result, type="gt")

CDM name,variable_name,estimate_name,estimate_value,cohort_name_reference,cohort_name_comparator
dbt-synthea,Number records,N,2,hypertension,sinusitis
dbt-synthea,Number subjects,N,2,hypertension,sinusitis
dbt-synthea,Days between cohort entries,Median [Q25 - Q75],"10,398.50 [8,451.00 - 12,346.00]",hypertension,sinusitis
dbt-synthea,Days between cohort entries,Range,"8,451.00 to 12,346.00",hypertension,sinusitis


In [17]:
fig = plot_cohort_timing(timing_result)
fig

## 8. Cohort Overlap

`summarise_cohort_overlap` quantifies overlap between cohorts (subjects appearing in multiple cohorts).

In [18]:
from omopy.characteristics import (
    summarise_cohort_overlap,
    table_cohort_overlap,
    plot_cohort_overlap,
)

overlap_result = summarise_cohort_overlap(cohort)
print(overlap_result)

SummarisedResult(12 rows, 1 result_id(s))


In [19]:
table_cohort_overlap(overlap_result, type="gt")

GT(_tbl_data=      CDM name estimate_name  \
0  dbt-synthea         N (%)   

  [header_name]variable_name\n[header_level]In both cohorts  \
0                                          2 (28.6%)          

  cohort_name_reference cohort_name_comparator  \
0          hypertension              sinusitis   

  [header_name]variable_name\n[header_level]Only in comparator cohort  \
0                                          1 (14.3%)                    

  [header_name]variable_name\n[header_level]Only in reference cohort  
0                                          4 (57.1%)                  , _body=<great_tables._gt_data.Body object at 0x72c90d6d0590>, _boxhead=Boxhead([ColInfo(var='CDM name', type=<ColInfoTypeEnum.default: 1>, column_label='CDM name', column_align='left', column_width=None), ColInfo(var='estimate_name', type=<ColInfoTypeEnum.default: 1>, column_label='estimate_name', column_align='left', column_width=None), ColInfo(var='[header_name]variable_name\n[header_level]In both cohorts', type=<ColInfoTypeEnum.default: 1>, column_label='In both cohorts', column_align='right', column_width=None), ColInfo(var='[header_name]variable_name\n[header_level]Only in comparator cohort', type=<ColInfoTypeEnum.default: 1>, column_label='Only in comparator cohort', column_align='right', column_width=None), ColInfo(var='[header_name]variable_name\n[header_level]Only in reference cohort', type=<ColInfoTypeEnum.default: 1>, column_label='Only in reference cohort', column_align='right', column_width=None), ColInfo(var='cohort_name_reference', type=<ColInfoTypeEnum.default: 1>, column_label='cohort_name_reference', column_align='left', column_width=None), ColInfo(var='cohort_name_comparator', type=<ColInfoTypeEnum.default: 1>, column_label='cohort_name_comparator', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x72c90d69c180>, _spanners=Spanners([SpannerInfo(spanner_id='variable_name', spanner_level=0, spanner_label='variable_name', spanner_units=None, spanner_pattern=None, vars=['[header_name]variable_name\n[header_level]In both cohorts', '[header_name]variable_name\n[header_level]Only in comparator cohort', '[header_name]variable_name\n[header_level]Only in reference cohort'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x72c90d5ff250>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x72c90d5ff350>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='CDM name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='estimate_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]variable_name\n[header_level]In both cohorts', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='cohort_name_reference', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='cohort_name_comparator', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]variable_name\n[header_level]Only in comparator cohort', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]variable_name\n[header_level]Only in reference cohort', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='CDM name', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, 

In [20]:
fig = plot_cohort_overlap(overlap_result)
fig

## 9. Large-Scale Characteristics

`summarise_large_scale_characteristics` performs feature extraction across the CDM.
Use the `event_in_window` or `episode_in_window` parameters to control which clinical events are captured.

In [21]:
from omopy.characteristics import (
    summarise_large_scale_characteristics,
    table_large_scale_characteristics,
    table_top_large_scale_characteristics,
    plot_large_scale_characteristics,
)

lsc_result = summarise_large_scale_characteristics(cohort)
print(lsc_result)

SummarisedResult(152 rows, 2 result_id(s))


In [22]:
table_large_scale_characteristics(lsc_result, type="gt")

cdm_name,variable_name,variable_level,estimate_value,table_name,type,analysis,cohort_name,concept_id
dbt-synthea,Chronic sinusitis,-Inf to -366,33.33,condition_occurrence,event,standard,hypertension,257012
dbt-synthea,Impacted molars,-Inf to -366,16.67,condition_occurrence,event,standard,hypertension,4055754
dbt-synthea,Chronic pain,-Inf to -366,16.67,condition_occurrence,event,standard,hypertension,436096
dbt-synthea,Essential hypertension,0 to 0,100.00,condition_occurrence,event,standard,hypertension,320128
dbt-synthea,Chronic pain,31 to 365,16.67,condition_occurrence,event,standard,hypertension,436096
dbt-synthea,Impacted molars,31 to 365,16.67,condition_occurrence,event,standard,hypertension,4055754
dbt-synthea,Fracture of rib,366 to Inf,16.67,condition_occurrence,event,standard,hypertension,4142905
dbt-synthea,Impaired glucose tolerance,366 to Inf,33.33,condition_occurrence,event,standard,hypertension,4311629
dbt-synthea,Hyperlipidemia,366 to Inf,33.33,condition_occurrence,event,standard,hypertension,432867
dbt-synthea,Miscarriage in first trimester,366 to Inf,16.67,condition_occurrence,event,standard,hypertension,4078393


In [23]:
# Show top concepts
table_top_large_scale_characteristics(lsc_result, top_concepts=10, type="gt")

cdm_name,variable_name,variable_level,table_name,type,analysis,cohort_name,concept_id,Frequency (%)
dbt-synthea,Essential hypertension,0 to 0,condition_occurrence,event,standard,hypertension,320128,100.0
dbt-synthea,Viral sinusitis,0 to 0,condition_occurrence,event,standard,sinusitis,40481087,100.0
dbt-synthea,lisinopril 10 MG Oral Tablet,366 to Inf,drug_exposure,event,standard,hypertension,19080128,83.33
dbt-synthea,Impaired glucose tolerance,-Inf to -366,condition_occurrence,event,standard,sinusitis,4311629,66.67
dbt-synthea,Impacted molars,-Inf to -366,condition_occurrence,event,standard,sinusitis,4055754,66.67
dbt-synthea,Essential hypertension,-Inf to -366,condition_occurrence,event,standard,sinusitis,320128,66.67
dbt-synthea,Anemia,-Inf to -366,condition_occurrence,event,standard,sinusitis,439777,66.67
dbt-synthea,Chronic pain,-Inf to -366,condition_occurrence,event,standard,sinusitis,436096,66.67
dbt-synthea,Chronic sinusitis,-Inf to -366,condition_occurrence,event,standard,sinusitis,257012,66.67
dbt-synthea,hydrochlorothiazide 25 MG Oral Tablet,0 to 0,drug_exposure,event,standard,hypertension,19078106,66.67


In [24]:
fig = plot_large_scale_characteristics(lsc_result)
fig

## 10. Cohort Codelist Summary

`summarise_cohort_codelist` summarises how concepts from the cohort's codelist appear in the CDM.
Note: there is no dedicated `table_codelist` or `plot_codelist` — use `table_characteristics` to display the result.

In [25]:
from omopy.characteristics import summarise_cohort_codelist

codelist_result = summarise_cohort_codelist(cohort)
print(codelist_result)

SummarisedResult(2 rows, 1 result_id(s))


In [26]:
# Use table_characteristics to render the codelist result
table_characteristics(codelist_result, type="gt")

GT(_tbl_data=  variable_name variable_level estimate_name  \
0       overall        overall    concept_id   
1       overall        overall    concept_id   

  [header_name]cdm_name\n[header_level]dbt-synthea   cohort_name  \
0                                       40,481,087     sinusitis   
1                                          320,128  hypertension   

  codelist_name codelist_type  
0     sinusitis   index event  
1  hypertension   index event  , _body=<great_tables._gt_data.Body object at 0x72c90d85fbf0>, _boxhead=Boxhead([ColInfo(var='variable_name', type=<ColInfoTypeEnum.default: 1>, column_label='variable_name', column_align='left', column_width=None), ColInfo(var='variable_level', type=<ColInfoTypeEnum.default: 1>, column_label='variable_level', column_align='left', column_width=None), ColInfo(var='estimate_name', type=<ColInfoTypeEnum.default: 1>, column_label='estimate_name', column_align='left', column_width=None), ColInfo(var='[header_name]cdm_name\n[header_level]dbt-synthea', type=<ColInfoTypeEnum.default: 1>, column_label='dbt-synthea', column_align='right', column_width=None), ColInfo(var='cohort_name', type=<ColInfoTypeEnum.default: 1>, column_label='cohort_name', column_align='left', column_width=None), ColInfo(var='codelist_name', type=<ColInfoTypeEnum.default: 1>, column_label='codelist_name', column_align='left', column_width=None), ColInfo(var='codelist_type', type=<ColInfoTypeEnum.default: 1>, column_label='codelist_type', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x72c90d6a3f00>, _spanners=Spanners([SpannerInfo(spanner_id='cdm_name', spanner_level=0, spanner_label='cdm_name', spanner_units=None, spanner_pattern=None, vars=['[header_name]cdm_name\n[header_level]dbt-synthea'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x72c90d5cb2b0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x72c90d8e6990>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_level', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='estimate_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]cdm_name\n[header_level]dbt-synthea', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='cohort_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='codelist_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='codelist_type', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_name', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_level', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='estimate_name', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weig

## 11. Working with SummarisedResult

`SummarisedResult` provides several methods for post-processing.

In [27]:
# .tidy() — flatten to a Polars DataFrame
tidy_df = char_result.tidy()
print(tidy_df)

shape: (68, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ result_id ┆ cdm_name  ┆ variable_ ┆ variable_ ┆ … ┆ result_ty ┆ package_n ┆ package_v ┆ cohort_n │
│ ---       ┆ ---       ┆ name      ┆ level     ┆   ┆ pe        ┆ ame       ┆ ersion    ┆ ame      │
│ i64       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1         ┆ dbt-synth ┆ Number    ┆           ┆ … ┆ summarise ┆ omopy.cha ┆ 0.1.0     ┆ sinusiti │
│           ┆ ea        ┆ records   ┆           ┆   ┆ _characte ┆ racterist ┆           ┆ s        │
│           ┆           ┆           ┆           ┆   ┆ ristics   ┆ ics       ┆           ┆          │
│ 1         ┆ dbt-synth ┆ Number    ┆           ┆ … ┆ summarise ┆ omopy.cha

In [28]:
# .pivot_estimates() — pivot estimate names into columns
pivoted = char_result.pivot_estimates()
print(pivoted)

shape: (16, 19)
┌───────────┬─────────────┬─────────────┬────────────┬───┬───────┬──────────┬─────────┬────────────┐
│ result_id ┆ cdm_name    ┆ group_name  ┆ group_leve ┆ … ┆ max   ┆ mean     ┆ sd      ┆ percentage │
│ ---       ┆ ---         ┆ ---         ┆ l          ┆   ┆ ---   ┆ ---      ┆ ---     ┆ ---        │
│ i64       ┆ str         ┆ str         ┆ ---        ┆   ┆ str   ┆ str      ┆ str     ┆ str        │
│           ┆             ┆             ┆ str        ┆   ┆       ┆          ┆         ┆            │
╞═══════════╪═════════════╪═════════════╪════════════╪═══╪═══════╪══════════╪═════════╪════════════╡
│ 1         ┆ dbt-synthea ┆ cohort_name ┆ sinusitis  ┆ … ┆ null  ┆ null     ┆ null    ┆ null       │
│ 1         ┆ dbt-synthea ┆ cohort_name ┆ sinusitis  ┆ … ┆ null  ┆ null     ┆ null    ┆ null       │
│ 1         ┆ dbt-synthea ┆ cohort_name ┆ sinusitis  ┆ … ┆ 14943 ┆ 14484.67 ┆ 659.36  ┆ null       │
│ 1         ┆ dbt-synthea ┆ cohort_name ┆ sinusitis  ┆ … ┆ 268   ┆ 160.67  

In [29]:
# .filter_settings() — filter by settings key-value pairs
filtered = char_result.filter_settings()
print(filtered)

SummarisedResult(68 rows, 1 result_id(s))


In [30]:
# .suppress() — suppress small cell counts for privacy
suppressed = char_result.suppress()
print(suppressed)

SummarisedResult(68 rows, 1 result_id(s))


In [31]:
# .split_group() — split group column into a DataFrame
split = char_result.split_group()
print(split)

shape: (68, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ result_id ┆ cdm_name  ┆ strata_na ┆ strata_le ┆ … ┆ estimate_ ┆ estimate_ ┆ estimate_ ┆ cohort_n │
│ ---       ┆ ---       ┆ me        ┆ vel       ┆   ┆ name      ┆ type      ┆ value     ┆ ame      │
│ i64       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1         ┆ dbt-synth ┆ overall   ┆ overall   ┆ … ┆ count     ┆ integer   ┆ 3         ┆ sinusiti │
│           ┆ ea        ┆           ┆           ┆   ┆           ┆           ┆           ┆ s        │
│ 1         ┆ dbt-synth ┆ overall   ┆ overall   ┆ … ┆ count     ┆ integer   ┆ 3         ┆ sinusiti │
│           ┆ ea        ┆           ┆           ┆   ┆           ┆          

In [32]:
# available_table_columns — see which columns can be used in table rendering
from omopy.characteristics import available_table_columns

columns = available_table_columns(char_result)
print(columns)

['cdm_name', 'cohort_name']


## 12. Using Mock Data

`mock_cohort_characteristics` generates a synthetic `SummarisedResult` without a database.
Useful for testing table and plot functions.

In [33]:
from omopy.characteristics import mock_cohort_characteristics

mock_result = mock_cohort_characteristics(n_cohorts=2, n_strata=0, seed=42)
print(type(mock_result))
print(mock_result)

<class 'omopy.generics.summarised_result.SummarisedResult'>
SummarisedResult(40 rows, 1 result_id(s))


In [34]:
# Tables and plots work on mock data too
table_characteristics(mock_result, type="gt")

GT(_tbl_data=        variable_name variable_level       estimate_name  \
0                 Sex           Male               N (%)   
1                 Sex         Female               N (%)   
2                 Sex           Male               N (%)   
3                 Sex         Female               N (%)   
4      Number records                                  N   
5     Number subjects                                  N   
6                 Sex           Male                   N   
7                 Sex         Female                   N   
8      Number records                                  N   
9     Number subjects                                  N   
10                Sex           Male                   N   
11                Sex         Female                   N   
12                Age                 Median [Q25 - Q75]   
13  Prior observation                 Median [Q25 - Q75]   
14                Age                 Median [Q25 - Q75]   
15  Prior observation                 Median [Q25 - Q75]   
16                Age                          Mean (SD)   
17  Prior observation                          Mean (SD)   
18                Age                          Mean (SD)   
19  Prior observation                          Mean (SD)   
20                Age                              Range   
21  Prior observation                              Range   
22                Age                              Range   
23  Prior observation                              Range   

   [header_name]cdm_name\n[header_level]mock_cdm cohort_name  
0                                    170 (45.1%)    cohort_1  
1                                    207 (54.9%)    cohort_1  
2                                    135 (31.5%)    cohort_2  
3                                    294 (68.5%)    cohort_2  
4                                            405    cohort_1  
5                                            377    cohort_1  
6                                            170    cohort_1  
7                                            207    cohort_1  
8                                            568    cohort_2  
9                                            429    cohort_2  
10                                           135    cohort_2  
11                                           294    cohort_2  
12                         36.00 [24.70 - 47.30]    cohort_1  
13                    848.84 [587.35 - 1,110.34]    cohort_1  
14                         38.48 [25.41 - 51.54]    cohort_2  
15                    734.24 [394.64 - 1,073.83]    cohort_2  
16                                 36.00 (11.30)    cohort_1  
17                               848.84 (261.50)    cohort_1  
18                                 38.48 (13.06)    cohort_2  
19                               734.24 (339.60)    cohort_2  
20                                 2.10 to 69.90    cohort_1  
21                            325.85 to 1,371.84    cohort_1  
22                                 0.00 to 77.67    cohort_2  
23                             55.05 to 1,413.43    cohort_2  , _body=<great_tables._gt_data.Body object at 0x72c90d6fa660>, _boxhead=Boxhead([ColInfo(var='variable_name', type=<ColInfoTypeEnum.default: 1>, column_label='variable_name', column_align='left', column_width=None), ColInfo(var='variable_level', type=<ColInfoTypeEnum.default: 1>, column_label='variable_level', column_align='left', column_width=None), ColInfo(var='estimate_name', type=<ColInfoTypeEnum.default: 1>, column_label='estimate_name', column_align='left', column_width=None), ColInfo(var='[header_name]cdm_name\n[header_level]mock_cdm', type=<ColInfoTypeEnum.default: 1>, column_label='mock_cdm', column_align='left', column_width=None), ColInfo(var='cohort_name', type=<ColInfoTypeEnum.default: 1>, column_label='cohort_name', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x72c984119a50>, _spanners=Spanners([SpannerInfo(spanner_id='cdm_name', spanner_level

In [35]:
fig = plot_characteristics(mock_result)
fig

## 13. Summary

In this notebook we covered the full `omopy.characteristics` workflow:

| Step | Functions |
|---|---|
| **Summarise** | `summarise_characteristics`, `summarise_cohort_count`, `summarise_cohort_attrition`, `summarise_cohort_timing`, `summarise_cohort_overlap`, `summarise_large_scale_characteristics`, `summarise_cohort_codelist` |
| **Table** | `table_characteristics`, `table_cohort_count`, `table_cohort_attrition`, `table_cohort_timing`, `table_cohort_overlap`, `table_large_scale_characteristics`, `table_top_large_scale_characteristics` |
| **Plot** | `plot_characteristics`, `plot_cohort_count`, `plot_cohort_attrition`, `plot_cohort_timing`, `plot_cohort_overlap`, `plot_large_scale_characteristics` |
| **SummarisedResult** | `.data`, `.tidy()`, `.pivot_estimates()`, `.filter_settings()`, `.suppress()`, `.split_group()` |
| **Utilities** | `available_table_columns`, `mock_cohort_characteristics` |

**Key points:**
- All summarise functions take a single `cohort` (`CohortTable`) as the first positional argument — the CDM is accessed via `cohort.cdm`
- There is no `min_cell_count` parameter on summarise functions — use `.suppress()` on the result instead
- `estimates` in `summarise_characteristics` is a `dict[str, tuple[str, ...]]`, not a list

**Next steps:**
- Add stratification with the `strata` parameter
- Combine multiple `SummarisedResult` objects
- Export results to CSV or other formats via `.tidy()`